# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print name and description from metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll print summary information for all record sets, including their `@id`, name, and contained fields.

In [ ]:
# List available record sets and their fields, referencing everything by @id
record_sets = dataset.metadata.record_sets

overview = []

for rs in record_sets:
    record_set_id = rs.id
    fields = getattr(rs, 'fields', [])
    print(f"RecordSet @id: {record_set_id}")
    print(f"  Name: {rs.name if hasattr(rs, 'name') else 'N/A'}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else 'N/A'}")
    print(f"  Fields:")
    for fld in fields:
        print(f"    Field @id: {fld.id}, Name: {fld.name if hasattr(fld, 'name') else 'N/A'}, DataType: {fld.data_type if hasattr(fld, 'data_type') else 'N/A'}")
    print("")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. We use the record set `@id`s listed above.

Note: If you want to analyze a particular record set, select its `@id` accordingly.

In [ ]:
# Extract the data from all available record sets
dataframes = {}
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for RecordSet {record_set_id}: {df.columns.tolist()}")
    print(df.head(), "\n")


## 4. Exploratory Data Analysis (EDA)
We will select a numeric field from one of the record sets for basic filtering and normalization.

Choose a record set and appropriate fields (by `@id`).

In [ ]:
# Example: Filtering records in a clinical features table
# Modify the record_set_id and field_id below to correspond to the intended table and numeric field

# For illustration, select the first record set and a field likely to be numeric (e.g., age at diagnosis)
record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None

if record_set_id and record_set_id in dataframes:
    df = dataframes[record_set_id]

    # Attempt to get a numeric field (choose by field @id)
    numeric_field_id = None
    field_ids = df.columns.tolist()

    # Try to choose a field likely to be numeric
    for field_id in field_ids:
        if 'age' in field_id.lower() or 'height' in field_id.lower():
            numeric_field_id = field_id
            break

    if numeric_field_id:
        # Filtering based on a threshold
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalizing the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to group by another field, e.g. 'infertility' status
        group_field = None
        for f in field_ids:
            if 'infertility' in f.lower():
                group_field = f
                break

        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field}:")
            print(grouped_df.head())
    else:
        print("No readily identifiable numeric field found in first record set.")
else:
    print("No record sets or dataframes loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we plot the distribution of the numeric field filtered above, and show a bar plot grouped by a categorical field (if available).

In [ ]:
import matplotlib.pyplot as plt

# Plotting if numeric field and filtered_df exist
if 'filtered_df' in locals() and numeric_field_id:
    plt.figure(figsize=(7, 4))
    filtered_df[numeric_field_id].hist(bins=10, alpha=0.7)
    plt.title(f"Distribution of {numeric_field_id} in filtered records")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Bar plot for group_field
    if group_field and group_field in filtered_df.columns:
        grouped = filtered_df.groupby(group_field)[numeric_field_id].mean()
        grouped.plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
We successfully loaded the FAIR^2 dataset, explored its structure by referencing all entities by their `@id`, extracted and processed data from its record sets, and performed basic exploratory analysis and visualization.

This dataset is small but demonstrates clear genotype–phenotype tabulations and supports rare disease clinical research. The limited scope and size (N=3, all female patients from a single center) were evident in the data and highlighted the need for caution when generalizing findings. The Croissant schema and `mlcroissant` library allow clear provenance and reproducible exploration.

Further investigation can expand on correlations, integrate imaging or sequence information, and enrich analysis as more data becomes available.